# Day 2.6 — Evaluate Retrieval Separately from Answers

When an answer is wrong, the first question is not "which prompt should I change?" but
"did the right evidence even reach the model?". A **golden set** - questions with known
expected evidence - lets us answer that with numbers instead of impressions.

## Before you begin

### Learning outcomes

- Score retrieval on its own, with the unanswerable case reported as n/a rather than a pass.
- Locate a failing case, change one layer, and re-measure the same set.

Architecture reference: [D07](../../diagrams/source/day_02.md).

### Expected observation

Under the offline hash embedder two of nine answerable cases miss their expected section;
switching to the trained embedder repairs them without touching a prompt.

### Modes

No API key needed: answers are scored with the deterministic offline generator.

## Concept briefing

## Diagnosing a bad answer

Use evidence in this order:

1. What exactly was the query?
2. Which chunks were retrieved and with what scores?
3. Does any retrieved chunk contain sufficient evidence?
4. Which chunk should have appeared according to the golden set?
5. If good evidence was present, did generation use it?
6. Did citation validation accept a source that was not actually retrieved?

If the correct evidence is absent, investigate ingestion, chunking, representation and
retrieval. If it is present but the answer is wrong, investigate context construction,
instructions, generation and validation. This separation prevents endless prompt edits
when the retriever never supplied the answer.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/knowledge_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Two indexes so we can compare representations on the same golden set.
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.embeddings import TokenHashEmbedder, load_embedder
from knowledge_agent.evaluation import (
    evaluate_answers,
    evaluate_retrieval,
    load_golden_set,
    render_table,
    summarize,
    summarize_detail,
    summarize_essential_terms,
)
from knowledge_agent.retrieval import VectorIndex

chunks = load_markdown_corpus(PROJECT_ROOT / "data" / "corpus")
cases = load_golden_set(PROJECT_ROOT / "data" / "golden_set.json")

hash_index = VectorIndex(TokenHashEmbedder())          # always available, fully deterministic
hash_index.add(chunks)

embedder, embedder_label = load_embedder()
semantic_index = None
if embedder_label == "semantic":
    semantic_index = VectorIndex(embedder)
    semantic_index.add(chunks)

print("Golden cases  :", len(cases))
print("Second index  :", "semantic" if semantic_index else "not available")

## Step 1 — Read the evaluation contract

Each case names the question, whether it is answerable, the expected source and section,
and the terms a correct answer must contain. The golden file is never indexed and never
shown to the model - it is the exam paper, not the textbook.

In [ ]:
for case in cases[:2]:
    print(case.model_dump())
    print()

unanswerable = [case for case in cases if not case.answerable]
print("answerable cases   :", len(cases) - len(unanswerable))
print("unanswerable cases :", len(unanswerable), "->", [case.id for case in unanswerable])
print("expected_source of the unanswerable case:", unanswerable[0].expected_source)

## Step 2 — Score retrieval alone

No generator is involved. For each case we ask: did the expected source appear in the
top-k, and did the expected *section* appear? The unanswerable case has no expected
evidence, so both columns print `n/a` - counting it as a hit would inflate the score.

In [ ]:
records_hash = evaluate_retrieval(hash_index, cases, top_k=3)
print(render_table(records_hash, ["id", "answerable", "source_hit", "section_hit", "expected_rank"]))
print()
print("rates :", summarize(records_hash, ["source_hit", "section_hit"]))
print("counts:", summarize_detail(records_hash, ["source_hit", "section_hit"]))

## Step 3 — Look at the misses, not the average

A rate is a pointer, not a diagnosis. Print the cases where the expected section never
reached the top-3 and compare what was retrieved with what was expected.

In [ ]:
misses = [record for record in records_hash if record["answerable"] and not record["section_hit"]]
print("cases missing their expected section:", [record["id"] for record in misses])

by_id = {case.id: case for case in cases}
for record in misses:
    case = by_id[record["id"]]
    print()
    print("case      :", case.id, "-", case.question)
    print("expected  :", case.expected_source, "|", case.expected_section)
    print("retrieved :")
    for chunk_id, section in zip(record["retrieved_ids"], record["retrieved_sections"]):
        print("   ", chunk_id, "|", section)
    print("source hit:", record["source_hit"], " section hit:", record["section_hit"])

## Step 4 — Change one layer and re-measure

The misses above are a *representation* problem: the questions paraphrase the documents.
Swap the embedder - nothing else - and run the identical set again.

In [ ]:
if semantic_index is None:
    print("Semantic embedder unavailable, so this comparison cannot run here.")
    print("Install sentence-transformers (or set EMBEDDER=semantic with network access)")
    print("and re-run: the two misses above are expected to disappear.")
else:
    records_semantic = evaluate_retrieval(semantic_index, cases, top_k=3)
    print(render_table(records_semantic, ["id", "answerable", "source_hit", "section_hit", "expected_rank"]))
    print()
    print("hash     rates:", summarize(records_hash, ["source_hit", "section_hit"]))
    print("semantic rates:", summarize(records_semantic, ["source_hit", "section_hit"]))
    print()
    for old, new in zip(records_hash, records_semantic):
        if old["section_hit"] != new["section_hit"]:
            print(f"{old['id']}: section_hit {old['section_hit']} -> {new['section_hit']}"
                  f"   (expected chunk rank {old['expected_rank']} -> {new['expected_rank']})")

## Step 5 — Does a bigger top-k fix everything?

Raising top-k can only help recall, but every extra chunk costs tokens and adds a
distractor the generator may quote instead. Measure the trade-off rather than guessing.

In [ ]:
def section_hit_rate(index, k):
    report = evaluate_retrieval(index, cases, top_k=k)
    answerable = [record for record in report if record["answerable"]]
    return sum(record["section_hit"] for record in answerable) / len(answerable)

print(f"{'top_k':8}{'hash':10}{'semantic':10}{'context sent':>14}")
for k in [1, 2, 3, 5]:
    semantic_value = f"{section_hit_rate(semantic_index, k):.2f}" if semantic_index else "n/a"
    average_chars = sum(len(chunk.text) for chunk in chunks) / len(chunks)
    print(f"{k:<8}{section_hit_rate(hash_index, k):<10.2f}{semantic_value:<10}{int(k * average_chars):>10} chars")

print()
print("Recall stops improving long before the cost does. top_k is a budget decision,")
print("not a quality dial.")

## Step 6 — Now score the answers, separately

Answer evaluation asks different questions: did it abstain when it should, did it cite the
expected source, did every citation survive validation, and did the text contain the
essential facts. Running it on the offline generator keeps this free.

In [ ]:
from knowledge_agent.assistant import KnowledgeAssistant
from knowledge_agent.generation import MockGroundedGenerator

assistant = KnowledgeAssistant(hash_index, MockGroundedGenerator(), top_k=3)
answers = evaluate_answers(assistant, cases)

print(render_table(answers, ["id", "answerable", "abstained", "abstention_correct",
                             "citation_correct", "citation_provenance_ok", "essential_term_coverage"]))
print()
fields = ["completed", "abstention_correct", "citation_correct", "citation_provenance_ok"]
print("rates :", summarize(answers, fields))
print("counts:", summarize_detail(answers, fields))
print("terms :", summarize_essential_terms(answers))

## Step 7 — Read the scorecard honestly

Two columns disagree on purpose, and that disagreement is the whole point of splitting
retrieval from answers.

In [ ]:
for record in answers:
    if record["essential_terms_total"] and record["essential_term_coverage"] == 0 and not record["abstained"]:
        print(record["id"], "cited the right source but contains none of the essential terms")
        print("   missing:", record["missing_terms"])
        print("   -> the answer quotes a chunk from the right FILE, from the wrong SECTION.")
        print("   -> compare with the retrieval table: same case, section_hit was NO.")
    if record["answerable"] and record["abstained"]:
        print(record["id"], "abstained although the corpus does contain the answer")
        print("   -> retrieval never supplied the section, so abstaining was the safest")
        print("      thing the generator could do with what it was given.")

### Try it yourself

Predict whether raising `top_k` to 5 repairs the missing sections under the hash embedder,
then check both the retrieval column and the answer column.

In [ ]:
# --- Worked solution ---
wide = evaluate_retrieval(hash_index, cases, top_k=5)
for old, new in zip(records_hash, wide):
    if old["section_hit"] != new["section_hit"]:
        print(f"{old['id']}: section_hit {old['section_hit']} -> {new['section_hit']}"
              f" (rank {old['expected_rank']} -> {new['expected_rank']})")

wide_answers = evaluate_answers(KnowledgeAssistant(hash_index, MockGroundedGenerator(), top_k=5), cases)
print()
print("answer rates at top_k=3:", summarize(answers, ["abstention_correct", "citation_correct"]))
print("answer rates at top_k=5:", summarize(wide_answers, ["abstention_correct", "citation_correct"]))
print()
print("More context can recover a missing section, but it also hands the generator more")
print("chances to quote the wrong one. Always re-measure both halves after a change.")

### Checkpoint

**1. Why is the unanswerable case reported as `n/a` instead of a hit?**

<details><summary>Show answer</summary>

Because it has no expected source or section, so "did we retrieve it?" has no answer. The
old version of this evaluator counted it as a pass, which quietly raised every retrieval
rate by ten percent and made a system with a real miss look better than it was. Rates are
now computed over the cases where the check applies, and the counts are printed beside
them.

</details>

**2. Retrieval scored 7/9 but the answers scored 9/10 for citations. Which number should
you act on?**

<details><summary>Show answer</summary>

The retrieval number. A citation can be "correct" at file level while the quoted section
is the wrong one - exactly what Step 7 prints for q01. Fixing generation cannot recover a
section that was never retrieved, so the retrieval miss is the defect to work on first.

</details>

### Recap

- **Limitation we saw:** an average hides which layer failed, and counting an
  inapplicable case as a pass inflates it further.
- **Layer we added:** separate retrieval and answer reports, with n/a for cases where a
  check does not apply and essential-term coverage beside the citation columns.
- **Evidence it worked:** the two hash-embedder misses are named, and swapping only the
  embedder repairs them on the identical set.

Work through [`reference/rag_failure_diagnosis.md`](../../reference/rag_failure_diagnosis.md)
next: it walks case q01 from this notebook through every layer, in order.